In [7]:
!pip install transformers
!pip install datasets
!pip install evaluate
!pip install peft
!pip install accelerate
!pip install bitsandbytes

In [8]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset")

In [9]:
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

In [20]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = ds["train"].to_pandas()

train_df, test_df = train_test_split(df, test_size=0.5, stratify=df["intent"])

from datasets import Dataset, DatasetDict
ds_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[:100]),
    "test": Dataset.from_pandas(test_df[:100])
})

print(ds_dataset)

DatasetDict({
    train: Dataset({
        features: ['tags', 'instruction', 'category', 'intent', 'response', '__index_level_0__'],
        num_rows: 100
    })
    test: Dataset({
        features: ['tags', 'instruction', 'category', 'intent', 'response', '__index_level_0__'],
        num_rows: 100
    })
})


In [21]:
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=len(df["response"].unique()))

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
from peft import get_peft_model, LoraConfig

# Define LoRA configuration for T5, targeting all layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "k_lin", "v_lin"])

# Convert T5 to a PEFT model with LoRA
lora_model = get_peft_model(model, lora_config)

In [23]:
# Sample a smaller subset of the dataset (for example, 10% of the original dataset)

# Tokenize function
def preprocess_function(examples):
    inputs = tokenizer(
        examples["instruction"],  # User input query
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    # Map responses to unique class labels
    label_map = {label: idx for idx, label in enumerate(sorted(set(df["response"])))}
    labels = [label_map[response] for response in examples["response"]]

    inputs["labels"] = labels
    return inputs


# Apply the tokenization to both 'train' and 'test' datasets
ds_dataset = ds_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [24]:
ds_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [25]:
ds_dataset

DatasetDict({
    train: Dataset({
        features: ['tags', 'instruction', 'category', 'intent', 'response', '__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
    test: Dataset({
        features: ['tags', 'instruction', 'category', 'intent', 'response', '__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [26]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [27]:
training_args = TrainingArguments(
    output_dir="./chatbot-finetuned",
    num_train_epochs=5,                # Increased epochs
    per_device_train_batch_size=2,    # Larger batch size
    per_device_eval_batch_size=2,
    weight_decay=0.1,                  # Adjusted weight decay
    learning_rate=5e-4,                # Lower learning rate
    evaluation_strategy="epoch",       # Keep evaluation strategy as epoch
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    gradient_accumulation_steps=2,     # Gradient accumulation
    max_grad_norm=1.0,                 # Gradient clipping
    fp16=True,                         # Mixed precision
    warmup_steps=50                   # Learning rate warmup
)


# Initialize the Trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,                 # Training arguments
    train_dataset=ds_dataset['train'],        # Training dataset
    eval_dataset=ds_dataset['test'],          # Evaluation dataset
    tokenizer=tokenizer,
    compute_metrics=compute_metrics# Tokenizer
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-27-f0a780189c35>:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [28]:
# Train the model
trainer.train()

wandb: WARNING Serializing object of type dict that is 1310808 bytes
wandb: WARNING Serializing object of type dict that is 1310808 bytes


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,10.150468,0.000000,0.000000,0.000000,0.000000
2,No log,10.150476,0.000000,0.000000,0.000000,0.000000
3,No log,10.150493,0.000000,0.000000,0.000000,0.000000
4,No log,10.150535,0.000000,0.000000,0.000000,0.000000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_

TrainOutput(global_step=30, training_loss=18.96334228515625, metrics={'train_runtime': 380.7666, 'train_samples_per_second': 1.313, 'train_steps_per_second': 0.079, 'total_flos': 20896794906624.0, 'train_loss': 18.96334228515625, 'epoch': 4.3076923076923075})

In [ ]:
# Evaluate the model after training
evaluation_results = trainer.evaluate()
print(evaluation_results)

{'eval_loss': 10.147699356079102, 'eval_runtime': 269.7881, 'eval_samples_per_second': 3.707, 'eval_steps_per_second': 0.463, 'epoch': 3.0}
